# Generate the synthetic portfolio

## How to use this notebook

Run cells in order with **Shift+Enter**. Change the settings below to create a
different, reproducible portfolio.

In [1]:
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook from the origination-scorecard-dev workspace")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())

In [2]:
n_accounts = 10_000
random_seed = 20_260_907
target_known_bad_rate = 0.04
target_inferred_rate = 0.02
broken_link_rate = 0.01

## Data-generation code

The generator is imported from the maintained package under `src/`, keeping this
notebook aligned with the tested implementation.

In [ ]:
import sys

source_directory = str(PROJECT_ROOT / "src")
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from origination_scorecard import (  # noqa: E402
    GeneratorConfig,
    generate_project_data,
    write_project_data,
)

In [4]:
config = GeneratorConfig(
    n_accounts=n_accounts,
    random_seed=random_seed,
    target_known_bad_rate=target_known_bad_rate,
    target_inferred_rate=target_inferred_rate,
    broken_link_rate=broken_link_rate,
)
outputs = generate_project_data(config)
write_project_data(outputs, PROJECT_ROOT)
{name: frame.shape for name, frame in outputs.items()}

{'applications': (9987, 38),
 'performance_12m': (10012, 13),
 'experian_consumer': (10000, 15),
 'experian_consumer_dictionary': (15, 5)}

## Check the generated files

The same run creates applications, 12-month performance and the Experian Consumer-style source.

In [5]:
generated_files = [
    PROJECT_ROOT / "data" / "applications.csv",
    PROJECT_ROOT / "data" / "performance_12m.csv",
    PROJECT_ROOT / "data" / "experian_consumer.csv",
    PROJECT_ROOT / "data" / "experian_consumer_dictionary.csv",
]
[
    (
        path.relative_to(PROJECT_ROOT).as_posix(),
        path.exists(),
        round(path.stat().st_size / 1_000_000, 2),
    )
    for path in generated_files
]

[('data/applications.csv', True, 2.62),
 ('data/performance_12m.csv', True, 0.38),
 ('data/experian_consumer.csv', True, 0.44),
 ('data/experian_consumer_dictionary.csv', True, 0.0)]

In [6]:
outputs["experian_consumer"].head()

,account_id,DelphiScore,E1A04,E1A09,E1B01,E1B07,E1B09,E1B12,E1B13,E1E01,E1E02,E1A01,E1A02,EA1C01,EA4Q01
0,ACC-10000000,825,0,0,1,0,5,1,0,1,1,0,0,N,N
1,ACC-10000001,871,0,0,2,0,3,1,0,2,3,0,0,N,N
2,ACC-10000002,881,0,0,1,0,1,0,0,2,2,0,0,N,N
3,ACC-10000003,800,0,1,0,1,2,0,0,0,1,0,0,N,N
4,ACC-10000004,841,0,1,2,1,2,0,0,2,2,0,0,N,T


## Experian Consumer source

The build also writes `data/experian_consumer.csv` and its data dictionary. Field names and meanings follow the Consumer schema in `RT-Howe/corezoid_poc`; values remain entirely synthetic teaching assumptions.